# 03. Chat Model 호출 추적

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. ChatOpenAI `invoke`, `stream`, `batch` 호출이 LangSmith에 어떻게 기록되는지 설명할 수 있어요
2. Trace에서 토큰 사용량(input/output tokens)을 확인하고 해석할 수 있어요
3. `ls_provider`, `ls_model_name` 메타데이터로 비용 추적을 설정할 수 있어요
4. `get_current_run_tree()`로 수동 토큰/비용 메타데이터를 Trace에 기록할 수 있어요

## 사전 지식

- `01-Setup-and-Overview`, `02-First-Trace` 노트북 완료
- `ChatOpenAI` 모델 기본 사용법
- LangSmith 프로젝트와 API Key 설정 완료

## 개념 설명: Chat Model Trace란?

LangSmith는 LangChain의 공식 Chat Model(ChatOpenAI, ChatAnthropic 등)을 자동으로 추적해요. 추적된 정보는 다음과 같아요:

| 항목 | 내용 |
|------|------|
| 입력(Input) | 전달된 메시지 목록 (HumanMessage, SystemMessage 등) |
| 출력(Output) | AI 응답 메시지 |
| 토큰 수 | `input_tokens`, `output_tokens`, `total_tokens` |
| 비용 | 모델 + 토큰 수 기반 자동 계산 |
| 지연 시간 | 시작~완료 시간, 스트리밍의 경우 `time_to_first_token` |

### LLM Trace가 다른 Trace와 다른 점

`run_type="llm"`으로 기록되면 LangSmith UI에서 토큰/비용 전용 패널이 표시돼요. 이를 위한 4가지 요소가 필요해요:

1. `run_type="llm"` 설정
2. 입출력이 LangChain 메시지 형식 (또는 OpenAI/Anthropic 형식)
3. 메타데이터에 `ls_provider`와 `ls_model_name` 포함
4. `usage_metadata`에 토큰 수 포함

> 🔑 **핵심 개념**: LangChain 공식 통합(ChatOpenAI, ChatAnthropic)을 사용하면 4가지 요소가 **자동**으로 처리돼요. 직접 구현한 모델 래퍼를 사용할 때만 수동 설정이 필요해요.

### Trace 흐름 다이어그램

```mermaid
flowchart LR
    A[코드: model.invoke] --> B[LangChain 통합 레이어]
    B --> C[OpenAI API 호출]
    C --> D[응답 수신]
    D --> B
    B --> E[LangSmith 자동 기록]

    E --> F1[입력 메시지]
    E --> F2[출력 메시지]
    E --> F3[토큰 수]
    E --> F4[비용 계산]
    E --> F5[지연 시간]

    classDef code fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404

    class A code
    class B,C,D process
    class E,F1,F2,F3,F4,F5 output
```

## 환경 설정

In [1]:
# ---------------------------------------------------
# 환경변수 로드
# ---------------------------------------------------
# .env 파일에서 API 키와 LangSmith 설정을 읽어와요
from dotenv import load_dotenv
import os

load_dotenv()

# LangSmith 추적 설정 확인
print(f"LANGSMITH_TRACING: {os.environ.get('LANGSMITH_TRACING', '')}")
print(f"LANGSMITH_PROJECT: {os.environ.get('LANGSMITH_PROJECT', '')}")
print(f"OPENAI_API_KEY: {os.environ.get('OPENAI_API_KEY', '')[:8]}...")

LANGSMITH_TRACING: 
LANGSMITH_PROJECT: 
OPENAI_API_KEY: sk-proj-...


In [2]:
# ---------------------------------------------------
# 이 노트북을 위한 LangSmith 프로젝트 설정
# ---------------------------------------------------
import os

# 이 노트북의 Trace는 별도 프로젝트에 기록해요
os.environ["LANGSMITH_PROJECT"] = "Ch03-ChatModel-Tracing"

print("프로젝트 설정 완료:", os.environ["LANGSMITH_PROJECT"])

프로젝트 설정 완료: Ch03-ChatModel-Tracing


## 1. invoke 호출 추적

`invoke`는 단일 동기 요청이에요. 요청과 응답이 하나의 Trace로 기록돼요.

> 🎯 **강의 포인트**: LangSmith UI에서 이 Trace를 찾아 **Inputs**, **Outputs**, **Token Usage** 섹션을 직접 확인해 보세요. 토큰 수와 예상 비용이 자동으로 표시돼요.

In [3]:
# ---------------------------------------------------
# invoke 호출: 단일 동기 요청
# ---------------------------------------------------
# ChatOpenAI를 사용하면 LangSmith 자동 추적이 활성화돼요
from langchain_openai import ChatOpenAI
from langchain.schema import HumanMessage, SystemMessage

# 기본 모델 초기화 (gpt-4o-mini: 비용 효율이 높아요)
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 단일 invoke 호출 - LangSmith에 하나의 Trace로 기록돼요
messages = [
    SystemMessage(content="당신은 친절한 AI 어시스턴트예요."),
    HumanMessage(content="LangSmith란 무엇인가요? 2문장으로 설명해 주세요.")
]

response = model.invoke(messages)

print("응답:", response.content)
print("\n--- 토큰 사용량 ---")
# response_metadata에서 토큰 정보를 확인할 수 있어요
print("입력 토큰:", response.response_metadata.get("token_usage", {}).get("prompt_tokens", "N/A"))
print("출력 토큰:", response.response_metadata.get("token_usage", {}).get("completion_tokens", "N/A"))
print("총 토큰:", response.response_metadata.get("token_usage", {}).get("total_tokens", "N/A"))

응답: LangSmith는 자연어 처리(NLP)와 관련된 다양한 작업을 지원하는 플랫폼으로, 개발자들이 언어 모델을 쉽게 구축하고 관리할 수 있도록 돕습니다. 이 도구는 데이터 수집, 모델 훈련, 평가 및 배포 과정을 간소화하여 사용자 경험을 향상시키는 데 중점을 두고 있습니다.

--- 토큰 사용량 ---
입력 토큰: 42
출력 토큰: 75
총 토큰: 117


**LangSmith UI 확인 방법**:
1. [smith.langchain.com](https://smith.langchain.com) 접속
2. 프로젝트 `LangSmith-Part2-ChatModelTracing` 선택
3. 방금 실행한 Trace 클릭
4. 오른쪽 패널에서 **Token Usage** 확인

## 2. stream 호출 추적

`stream`은 응답을 조각(chunk) 단위로 받아요. LangSmith는 스트리밍 완료 후 전체 Trace를 기록하고, 특히 `time_to_first_token`(첫 토큰 도착 시간)을 측정해요.

> 🔑 **핵심 개념**: `stream` Trace에는 `time_to_first_token` 메트릭이 추가돼요. 사용자 체감 응답 속도를 평가할 때 중요한 지표예요.

In [4]:
# ---------------------------------------------------
# stream 호출: 실시간 스트리밍 응답
# ---------------------------------------------------
# 스트리밍은 응답을 조각 단위로 받아요
# LangSmith는 완료 후 전체 응답을 하나의 Trace로 기록해요

print("스트리밍 응답: ", end="", flush=True)

full_content = ""
for chunk in model.stream("파이썬의 장점을 3가지로 설명해 주세요."):
    print(chunk.content, end="", flush=True)  # 실시간 출력
    full_content += chunk.content

print()  # 줄바꿈
print(f"\n총 응답 길이: {len(full_content)}자")
print("\nLangSmith UI에서 이 Trace의 'time_to_first_token' 값을 확인해 보세요!")

스트리밍 응답: 파이썬의 장점은 다음과 같습니다:

1. **간결하고 읽기 쉬운 문법**: 파이썬은 코드가 간결하고 명확하여, 초보자도 쉽게 이해하고 배울 수 있습니다. 들여쓰기를 통해 코드 블록을 구분하므로 가독성이 높아지고, 이는 유지보수와 협업에 유리합니다.

2. **풍부한 라이브러리와 프레임워크**: 파이썬은 데이터 분석, 웹 개발, 인공지능, 머신러닝 등 다양한 분야에서 사용할 수 있는 방대한 라이브러리와 프레임워크를 제공합니다. 예를 들어, NumPy, Pandas, TensorFlow, Django 등이 있어 개발자가 필요한 기능을 쉽게 구현할 수 있습니다.

3. **활발한 커뮤니티와 지원**: 파이썬은 전 세계적으로 많은 사용자와 개발자 커뮤니티가 있어, 문제 해결이나 정보 공유가 용이합니다. 다양한 온라인 자료, 튜토리얼, 포럼 등이 있어 학습과 개발에 큰 도움이 됩니다.

총 응답 길이: 439자

LangSmith UI에서 이 Trace의 'time_to_first_token' 값을 확인해 보세요!


## 3. batch 호출 추적

`batch`는 여러 입력을 병렬로 처리해요. LangSmith에서는 각 요청이 독립적인 Trace로 기록돼요.

> 💡 **실무 팁**: `batch`를 사용하면 여러 입력을 동시에 처리해서 총 처리 시간을 줄일 수 있어요. LangSmith에서 각 Trace의 지연 시간을 비교하면 병렬 처리 효과를 확인할 수 있어요.

In [5]:
# ---------------------------------------------------
# batch 호출: 여러 입력 병렬 처리
# ---------------------------------------------------
# batch는 내부적으로 여러 요청을 병렬로 보내요
# LangSmith에서 각 요청이 별도 Trace로 기록돼요

questions = [
    "파이썬이란 무엇인가요?",
    "LangChain이란 무엇인가요?",
    "벡터 데이터베이스란 무엇인가요?"
]

print("batch 호출 시작 (3개 질문 병렬 처리)...")

# batch로 3개 질문을 동시에 처리해요
responses = model.batch(questions)

print("\n--- 응답 결과 ---")
for i, (q, r) in enumerate(zip(questions, responses), 1):
    print(f"\n질문 {i}: {q}")
    print(f"답변: {r.content[:100]}...")  # 처음 100자만 표시

batch 호출 시작 (3개 질문 병렬 처리)...

--- 응답 결과 ---

질문 1: 파이썬이란 무엇인가요?
답변: 파이썬(Python)은 고급 프로그래밍 언어로, 1991년 귀도 반 로섬(Guido van Rossum)에 의해 처음 개발되었습니다. 파이썬은 코드의 가독성이 높고, 문법이 간결하...

질문 2: LangChain이란 무엇인가요?
답변: LangChain은 자연어 처리(NLP)와 관련된 다양한 작업을 수행하기 위해 설계된 프레임워크입니다. 주로 대화형 AI 애플리케이션, 챗봇, 정보 검색 시스템 등을 구축하는 데 ...

질문 3: 벡터 데이터베이스란 무엇인가요?
답변: 벡터 데이터베이스는 고차원 벡터를 저장하고 검색하는 데 최적화된 데이터베이스입니다. 이러한 벡터는 일반적으로 기계 학습 모델, 특히 딥러닝 모델을 통해 생성된 데이터 표현을 나타냅...


### invoke vs stream vs batch 비교

| 호출 방식 | Trace 개수 | 특징 | 주요 메트릭 |
|-----------|-----------|------|-------------|
| `invoke` | 1개 | 단일 동기 요청 | 총 지연 시간, 토큰 수 |
| `stream` | 1개 | 스트리밍, 점진적 응답 | `time_to_first_token` 추가 |
| `batch` | N개 | 여러 요청 병렬 처리 | 각 요청 독립 Trace |

## 4. 모델별 응답 시간 비교

동일한 질문을 다른 모델로 실행하면 응답 시간과 토큰 비용 차이를 LangSmith에서 시각적으로 비교할 수 있어요.

> 🎯 **강의 포인트**: LangSmith에서 두 Trace를 나란히 비교(Compare 기능)해 보세요. 응답 품질, 토큰 수, 비용, 지연 시간의 트레이드오프를 확인할 수 있어요.

In [6]:
# ---------------------------------------------------
# 모델별 응답 시간 및 토큰 비교
# ---------------------------------------------------
# 두 모델로 같은 질문을 보내고 LangSmith에서 비교해요
import time

# gpt-4o-mini: 빠르고 저렴해요
model_mini = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# gpt-4o: 더 강력하지만 느리고 비싸요
model_4o = ChatOpenAI(model="gpt-4o", temperature=0)

# 비교할 질문
test_question = "머신러닝과 딥러닝의 차이점을 간단히 설명해 주세요."

# gpt-4o-mini 실행
start = time.time()
response_mini = model_mini.invoke(test_question)
time_mini = time.time() - start

# gpt-4o 실행
start = time.time()
response_4o = model_4o.invoke(test_question)
time_4o = time.time() - start

print("=== gpt-4o-mini ===")
print(f"응답 시간: {time_mini:.2f}초")
tokens_mini = response_mini.response_metadata.get("token_usage", {})
print(f"토큰 사용량: 입력 {tokens_mini.get('prompt_tokens', 'N/A')}, 출력 {tokens_mini.get('completion_tokens', 'N/A')}")
print(f"응답: {response_mini.content[:200]}")

print("\n=== gpt-4o ===")
print(f"응답 시간: {time_4o:.2f}초")
tokens_4o = response_4o.response_metadata.get("token_usage", {})
print(f"토큰 사용량: 입력 {tokens_4o.get('prompt_tokens', 'N/A')}, 출력 {tokens_4o.get('completion_tokens', 'N/A')}")
print(f"응답: {response_4o.content[:200]}")

=== gpt-4o-mini ===
응답 시간: 5.44초
토큰 사용량: 입력 27, 출력 290
응답: 머신러닝과 딥러닝은 모두 인공지능(AI)의 하위 분야이지만, 그 접근 방식과 사용되는 기술에서 차이가 있습니다.

1. **머신러닝 (Machine Learning)**:
   - 머신러닝은 데이터에서 패턴을 학습하여 예측이나 결정을 내리는 알고리즘을 개발하는 분야입니다.
   - 주로 특성(피처)을 수동으로 선택하고, 이를 기반으로 모델을 훈련합니다.
 

=== gpt-4o ===
응답 시간: 5.19초
토큰 사용량: 입력 27, 출력 342
응답: 머신러닝과 딥러닝은 모두 인공지능의 하위 분야이지만, 그 구조와 접근 방식에서 차이가 있습니다.

1. **머신러닝**:
   - 머신러닝은 데이터에서 패턴을 학습하여 예측이나 결정을 내리는 알고리즘을 개발하는 분야입니다.
   - 주로 특징(feature)을 수동으로 추출하고, 이를 기반으로 모델을 학습시킵니다.
   - 선형 회귀, 로지스틱 회귀, 의사


## 5. 비용 추적: ls_provider와 ls_model_name

LangSmith가 비용을 자동으로 계산하려면 `ls_provider`와 `ls_model_name`이 필요해요. ChatOpenAI 같은 공식 통합은 이를 자동으로 설정하지만, 직접 `@traceable`을 사용할 때는 수동으로 설정해야 해요.

> ⚠️ **자주 하는 실수**: `ls_model_name`을 잘못 입력하면 비용 계산이 안 돼요. OpenAI의 경우 `gpt-4o-mini`, `gpt-4o`처럼 **정확한 모델 ID**를 사용해야 해요.

In [7]:
# ---------------------------------------------------
# 비용 추적: ls_* 메타데이터 설정
# ---------------------------------------------------
# @traceable을 사용한 직접 구현 시 비용 추적 방법이에요
from langsmith import traceable, get_current_run_tree
import openai

# OpenAI 클라이언트 직접 사용 예제 (공식 통합 없이 직접 구현)
@traceable(
    run_type="llm",  # LLM 타입으로 지정해야 토큰/비용 패널이 표시돼요
    name="custom-gpt-call",
    metadata={
        "ls_provider": "openai",       # 비용 계산에 필요한 공급자 정보
        "ls_model_name": "gpt-4o-mini" # 정확한 모델 ID를 입력해야 해요
    }
)
def custom_llm_call(messages: list) -> dict:
    """OpenAI API를 직접 호출하는 커스텀 함수예요"""
    # OpenAI API 직접 호출 (LangChain 없이)
    client = openai.OpenAI()
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        temperature=0
    )

    # 수동으로 토큰 사용량을 Trace에 기록해요
    run = get_current_run_tree()
    if run:
        run.set(usage_metadata={
            "input_tokens": response.usage.prompt_tokens,
            "output_tokens": response.usage.completion_tokens,
            "total_tokens": response.usage.total_tokens
        })

    # LangChain 메시지 형식으로 반환해요
    return {"role": "assistant", "content": response.choices[0].message.content}

# 실행
result = custom_llm_call([
    {"role": "user", "content": "RAG 파이프라인이란 무엇인가요? 한 문장으로 설명해 주세요."}
])
print("응답:", result["content"])
print("\nLangSmith에서 이 Trace의 'Token Usage' 섹션을 확인해 보세요!")

응답: RAG 파이프라인은 Retrieval-Augmented Generation의 약자로, 외부 데이터베이스에서 정보를 검색한 후 이를 활용하여 자연어 생성 모델이 더 정확하고 풍부한 응답을 생성하는 방식입니다.

LangSmith에서 이 Trace의 'Token Usage' 섹션을 확인해 보세요!


## 6. 직접 비용 지정: input_cost와 output_cost

토큰 수 대신 **직접 비용을 달러로 지정**할 수도 있어요. 제3자 모델이나 커스텀 요금제를 사용할 때 유용해요.

> 💡 **실무 팁**: OpenAI 공식 통합을 사용하면 비용이 자동으로 계산되지만, 기업 특별 요금제나 자체 모델 배포 시에는 직접 비용을 입력할 수 있어요.

In [8]:
# ---------------------------------------------------
# 직접 비용 지정 방법
# ---------------------------------------------------
# input_cost / output_cost로 비용을 직접 달러로 지정해요

@traceable(
    run_type="llm",
    name="custom-cost-tracking"
)
def llm_with_manual_cost(prompt: str) -> str:
    """직접 비용을 지정하는 예제예요"""
    client = openai.OpenAI()
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    # 토큰 기반 비용 직접 계산
    # gpt-4o-mini 요금: 입력 $0.15/1M, 출력 $0.60/1M (예시)
    prompt_tokens = response.usage.prompt_tokens
    completion_tokens = response.usage.completion_tokens

    input_cost = prompt_tokens * 0.00000015    # $0.15/1M 토큰
    output_cost = completion_tokens * 0.00000060  # $0.60/1M 토큰

    # get_current_run_tree()로 현재 Trace에 비용 메타데이터를 추가해요
    run = get_current_run_tree()
    if run:
        run.set(usage_metadata={
            "input_cost": input_cost,    # 달러 단위
            "output_cost": output_cost,  # 달러 단위
            "total_cost": input_cost + output_cost,
            "input_tokens": prompt_tokens,
            "output_tokens": completion_tokens
        })

    return response.choices[0].message.content

# 실행
answer = llm_with_manual_cost("임베딩(Embedding)이란 무엇인가요?")
print("응답:", answer)
print("\nLangSmith Trace에서 직접 지정한 비용을 확인할 수 있어요!")

응답: 임베딩(Embedding)은 고차원 데이터를 저차원 공간에 매핑하는 방법으로, 주로 기계 학습과 자연어 처리(NLP) 분야에서 사용됩니다. 임베딩의 주요 목적은 데이터의 의미를 보존하면서 더 효율적으로 표현할 수 있도록 하는 것입니다.

예를 들어, 단어 임베딩(Word Embedding)은 단어를 고차원 벡터 공간의 저차원 벡터로 변환하여 단어 간의 의미적 유사성을 반영합니다. 대표적인 단어 임베딩 기법으로는 Word2Vec, GloVe, FastText 등이 있습니다. 이러한 기법들은 단어 간의 관계를 수치적으로 표현하여, 유사한 의미를 가진 단어들이 가까운 벡터로 위치하도록 합니다.

임베딩은 또한 이미지, 그래프, 사용자 행동 등 다양한 데이터 유형에 적용될 수 있으며, 이를 통해 기계 학습 모델의 성능을 향상시키고, 데이터의 복잡성을 줄이는 데 기여합니다.

LangSmith Trace에서 직접 지정한 비용을 확인할 수 있어요!


## 7. 실습: 다양한 설정으로 비교 실험

아래 코드를 수정해서 직접 실험해 보세요!

In [9]:
# ============================================================
# TODO: 아래 설정을 바꿔가며 LangSmith Trace 변화를 관찰해 보세요
# 힌트: temperature, max_tokens를 변경해 보고 Trace 메타데이터가 어떻게 달라지는지 확인해요
# 예상 결과: 다른 temperature 값으로 응답 다양성이 변하고, Trace에 ls_invocation_params가 기록돼요
# ============================================================

from langchain_core.tracers.langchain import wait_for_all_tracers

# 설정을 변경해 보세요
TEMPERATURE = 0.7    # 0 ~ 2 사이의 값으로 바꿔 보세요
MAX_TOKENS = 200     # 응답 최대 토큰 수를 변경해 보세요
TEST_PROMPT = "인공지능이 세상을 바꿀 방식 3가지를 알려 주세요."  # 질문도 바꿔 보세요

# 설정 적용
experiment_model = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS
)

response = experiment_model.invoke(TEST_PROMPT)
print(f"temperature={TEMPERATURE}, max_tokens={MAX_TOKENS}")
print(f"응답: {response.content}")

# 모든 Trace 전송 완료까지 대기해요
wait_for_all_tracers()
print("\nLangSmith Trace 기록 완료!")

temperature=0.7, max_tokens=200
응답: 인공지능(AI)은 다양한 방식으로 세상을 변화시키고 있습니다. 그 중에서 세 가지 주요 방식을 소개합니다.

1. **의료 혁신**: 인공지능은 진단 및 치료 과정에서 큰 변화를 일으키고 있습니다. AI는 방대한 양의 의료 데이터를 분석하여 질병을 조기에 발견하고, 개인 맞춤형 치료법을 제안할 수 있습니다. 예를 들어, 이미지 인식 기술을 활용하여 X-ray나 MRI 이미지를 분석하고, 암과 같은 질병을 더 정확하게 진단하는 데 도움을 줄 수 있습니다.

2. **자동화와 효율성**: 인공지능은 다양한 산업에서 작업을 자동화하여 효율성을 높이고 비용을 절감할 수 있습니다. 제조업, 물류, 고객 서비스 등에서 AI 기반의 로봇이나 챗봇이 인간의 작업을 보조하거나 대체함으로써 생산성을 향상시키

LangSmith Trace 기록 완료!


## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **자동 추적**: ChatOpenAI 사용 시 토큰, 비용, 지연 시간이 LangSmith에 자동 기록돼요
- **invoke vs stream vs batch**: 각 호출 방식에 따라 Trace 구조가 달라요. `stream`은 `time_to_first_token` 추가
- **ls_provider / ls_model_name**: 비용 계산에 필요한 메타데이터예요. 공식 통합은 자동 설정
- **get_current_run_tree()**: 현재 실행 중인 Trace 객체를 가져와 수동으로 메타데이터를 추가할 수 있어요
- **직접 비용 지정**: `usage_metadata`에 `input_cost` / `output_cost`를 달러로 직접 지정 가능해요

## 다음 노트북 예고

다음 `04-Chain-and-LCEL-Tracing.ipynb`에서는 **LCEL 체인의 계층적 Trace 구조**를 배워요. `|` 연산자로 연결된 각 컴포넌트가 어떻게 부모-자식 관계로 기록되는지 살펴볼게요.